# Day 5: Markets as Mean-Field Games

*Alpha Flow Research · HongJin HE · July 2026*

---

## From Single Agent to Market

Classical finance assumes prices are set by a single "representative agent." This is wrong:
- Markets are played by **thousands of heterogeneous agents** with different information, capital, and objectives
- Each agent's decision **affects** the prices that other agents face
- This is a game, not an optimization problem

**Mean-Field Game (MFG)** theory (Lasry & Lions, 2007) provides the right framework:
> When $N \to \infty$ agents interact, each agent's optimal strategy depends only on the **distribution** of all other agents' states, not on each individual.

## The E-Game-C MFG Formulation

Each of $N \to \infty$ representative agents solves:
$$\inf_{\alpha \in \mathcal{A}} \mathbb{E}\left[\int_0^T f(z_t, \alpha_t, m_t)\,dt + g(z_T, m_T)\right]$$
subject to:
$$dz_t = b(z_t, \alpha_t, m_t)\,dt + \sigma_\tau\,dW_t + \int_{\mathbb{R}^d} \xi\,\tilde{N}_\eta(dt, d\xi)$$

where $m_t = \text{Law}(z_t)$ is the **mean field** — the empirical distribution of all agents.

**Nash Equilibrium**: $(V^*, m^*)$ satisfying the coupled HJB + Fokker-Planck system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

np.random.seed(42)

# ── Toy MFG: 1D latent space, quadratic cost ──────────────────────────────────
# True equilibrium: all agents converge to z* = 0 with drift a*(z) = -z
# V*(z) = -z² / (2γκ)  →  a*(z) = ∇V* / (2γκ) = -z / (2γκ)²  → -z (normalized)

gamma, kappa = 2.0, 0.01
sigma_tau = 0.3
dt = 0.05
n_particles = 500
n_outer = 30
n_inner = 10

def V_star(z): return -z**2   # true quadratic value function
def grad_V(z): return -2*z
def optimal_control(z): return grad_V(z) / (2 * gamma * kappa)

# Run Neural Fictitious Play
particles = np.random.randn(n_particles) * 2.0   # start dispersed
histories = [particles.copy()]
residuals = []

for n in range(n_outer):
    prev_mean = particles.mean()
    for _ in range(n_inner):
        alpha = optimal_control(particles)
        noise = np.random.randn(n_particles) * sigma_tau * np.sqrt(dt)
        particles = particles + alpha * dt + noise
    residuals.append(abs(particles.mean() - prev_mean))
    if n in [0, 4, 9, 19, 29]:
        histories.append(particles.copy())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Agent distribution evolution
ax = axes[0]
colors_ev = plt.cm.Blues(np.linspace(0.3, 1.0, len(histories)))
labels_ev = ['Initial', 'Iter 5', 'Iter 10', 'Iter 20', 'Iter 30', 'Final']
bins = np.linspace(-6, 6, 50)
for i, (hist, color) in enumerate(zip(histories, colors_ev)):
    ax.hist(hist, bins=bins, density=True, alpha=0.6, color=color,
            label=labels_ev[i] if i < len(labels_ev) else '')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Nash equilibrium z*=0')
ax.set_xlabel('Latent state z'); ax.set_ylabel('Density')
ax.set_title('Mean-Field Distribution Converging\nto Nash Equilibrium', fontweight='bold')
ax.legend(fontsize=8)

# Middle: Convergence curve
ax = axes[1]
ax.semilogy(range(len(residuals)), residuals, 'o-', color='#2E86AB', linewidth=2, markersize=4)
ax.axhline(0.005, color='red', linestyle='--', label='Convergence threshold δ=0.005')
ax.set_xlabel('Outer iteration'); ax.set_ylabel('W₂ residual (approx)')
ax.set_title('Neural Fictitious Play Convergence\n(Theorem 2: exponential rate)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Optimal control drift field
ax = axes[2]
z_range = np.linspace(-4, 4, 100)
drift = optimal_control(z_range)
value = V_star(z_range)

ax2_twin = ax.twinx()
ax.plot(z_range, drift,  color='#2E86AB', linewidth=2.5, label='Optimal drift a*(z) = ∇V*/(2γκ)')
ax2_twin.plot(z_range, value, color='#C73E1D', linewidth=2.0, linestyle='--', label='Value function V*(z)')
ax.axhline(0, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
ax.set_xlabel('Latent state z'); ax.set_ylabel('Drift a*(z)', color='#2E86AB')
ax2_twin.set_ylabel('Value V*(z)', color='#C73E1D')
ax.set_title('HJB Solution: Optimal Control Field\na*(z) = ∇V*/(2γκ) [Controller C]', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/fig_mfg_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final mean field converged to: μ = {particles.mean():.4f} (target: 0.0)')

## The Three Theorems

**Theorem 1** (Cramér-Rao): Dual noise gives irreducible prediction floors. Behavioral component $\nu_\eta(\mathbb{R})$ cannot be reduced by more data.

**Theorem 2** (Unique Nash Equilibrium): Under Lasry-Lions monotonicity, the MFG system has a **unique** Nash equilibrium $(V^*, m^*)$, and Neural Fictitious Play converges exponentially:
$$W_2(m^{(n)}, m^*) \leq C \rho^n, \quad \rho \in (0,1)$$

**Theorem 3** (Lyapunov Stability): Under equilibrium policy $\alpha^*$, the market admits a **unique stationary distribution** $\pi^*$ with exponential convergence:
$$\|\text{Law}(z_t) - \pi^*\|_{\text{TV}} \leq K e^{-ct}$$

This is why the world model is predictive: the market has a **center of gravity** (the Nash equilibrium) that it perpetually returns to, even after shocks.

**Next**: Day 6 — MFC/MFG Hierarchy (`day06_mfc_mfg_hierarchy.ipynb`)